# Evolução da umidade relativa — SONDA-MD

Gera a média diária da umidade relativa da base tratada da SONDA-MD. Os dias com menos de 24 observações são destacados; nenhum valor ausente é interpolado.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

def localizar_dados_tratados():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidato = base / '2026' / 'dados_tratados'
        if (candidato / 'sonda_md_horario_padronizado.csv').exists():
            return candidato.resolve()
    raise FileNotFoundError('Não foi encontrada a pasta 2026/dados_tratados.')

PASTA_DADOS = localizar_dados_tratados()
PASTA_SAIDA = PASTA_DADOS.parent / 'resultados' / 'figuras' / 'comparacao_bases'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO = PASTA_DADOS / 'sonda_md_horario_padronizado.csv'
dados = pd.read_csv(ARQUIVO, parse_dates=['data_hora'])
dados['umidade_relativa'] = pd.to_numeric(dados['umidade_relativa'], errors='coerce')
dados = (dados[['data_hora', 'umidade_relativa']]
         .dropna(subset=['data_hora'])
         .sort_values('data_hora')
         .drop_duplicates('data_hora')
         .set_index('data_hora'))

diaria = dados['umidade_relativa'].resample('D').agg(['mean', 'count'])
dias_parciais = diaria.index[diaria['count'] < 24]
print(f'Registros horários: {len(dados):,}')
print(f'Período: {dados.index.min()} a {dados.index.max()}')
print(f'Valores ausentes de umidade: {dados.umidade_relativa.isna().sum():,}')
print(f'Dias com cobertura parcial: {len(dias_parciais):,}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.2), dpi=150)
diaria['mean'].plot(ax=ax, color='#1f5a85', linewidth=0.8)
for dia in dias_parciais:
    ax.axvspan(dia, dia + pd.Timedelta(days=1), color='#e6a700', alpha=0.22, linewidth=0)
ax.set_title('Umidade relativa da SONDA-MD (média diária)')
ax.set_xlabel('Data')
ax.set_ylabel('Umidade relativa (%)')
ax.legend(handles=[Patch(facecolor='#e6a700', alpha=0.22, label='Dia com cobertura parcial')], loc='best', fontsize=8)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

ARQUIVO_FIGURA = PASTA_SAIDA / 'sonda_md_umidade_evolucao.png'
fig.savefig(ARQUIVO_FIGURA, dpi=300, bbox_inches='tight')
print(f'Figura salva em: {ARQUIVO_FIGURA}')
plt.show()